# Inferencia autoregresiva Kimi K3

Este notebook reconstruye la arquitectura desde YAML, carga un checkpoint entrenado y genera usando `prefill` seguido de `decode_step` con el cache KDA/MLA nativo. No existe todavía un checkpoint incluido en el repositorio, por lo que las celdas quedan sin ejecutar.

In [ ]:
from pathlib import Path

from configuration import resolve_kimi_pipeline_profile
from data import load_tokenizer_from_data_yaml
from inference import (
    ModelLoadConfig,
    inference_autoregressive,
    load_generation_config,
    load_kimi_checkpoint,
)

In [ ]:
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
PROFILE = resolve_kimi_pipeline_profile(ROOT / 'config/kimi_full_pipeline/low_gpu')
DATA_YAML = PROFILE.data
MODEL_YAML = PROFILE.model
INFERENCE_YAML = ROOT / 'config/inference/greedy.yaml'
CHECKPOINT = ROOT / 'checkpoints/t4_15gb/kimi_k3_t4_15gb_epoch_0002.pt'

## 1. Tokenizer y checkpoint

El tokenizer se reconstruye desde el YAML de datos. Para Hugging Face debe existir el archivo cacheado durante la preparación de datos.

In [ ]:
tokenizer = load_tokenizer_from_data_yaml(DATA_YAML)
loaded = load_kimi_checkpoint(
    MODEL_YAML,
    CHECKPOINT,
    tokenizer=tokenizer,
    load_config=ModelLoadConfig(device='cuda', precision='fp16'),
)
model = loaded.model

## 2. Generación

La función maestra tokeniza, hace un único prefill, decodifica token a token con cache y convierte la salida nuevamente a texto.

In [ ]:
generation_config = load_generation_config(INFERENCE_YAML)
output = inference_autoregressive(
    model,
    'Once upon a time',
    tokenizer=tokenizer,
    generation_config=generation_config,
)
print(output.completion_text)

In [ ]:
{
    'finish_reason': output.finish_reason,
    'generated_tokens': output.generated_tokens,
    'tokens_per_second': output.tokens_per_second,
    'cache': output.cache_stats,
}